In [ ]:
#random forest implementation here 
#using the LOSO method which I have heard is the standard for PAMPA2 

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import shuffle

In [4]:
#first we need to take snapshots of the each data set with the sliding window 
def sliding_window_features(df, window_size=100, step=50):
    feature_cols = df.columns.drop("activityID")
    X, y = [], []
    
    for start in range(0, len(df) - window_size + 1, step):
        window = df.iloc[start:start + window_size]
      
        label = window["activityID"].mode()[0]
      
        features = []
        for col in feature_cols:
            features += [
                window[col].mean(),
                window[col].std(),
                window[col].min(),
                window[col].max(),
            ]
        
        X.append(features)
        y.append(label)
    
    return np.array(X), np.array(y)
  

In [5]:
import os
os.makedirs(r"C:\Users\User\OneDrive - UBC\Desktop\cpen 355\Final_project\CPEN355_FinalProject\preprocessing\data", exist_ok=True)

for i in range(1, 9):
    globals()[f"data_{i}"] = pd.read_pickle(rf"C:\Users\User\OneDrive - UBC\Desktop\cpen 355\Final_project\CPEN355_FinalProject\preprocessing\data\data_{i}.pkl")

X_all = {}
y_all = {}

for i in range(1, 9):
    df = globals()[f"data_{i}"]
    X, y = sliding_window_features(df, window_size=100, step=50)
    X_all[i] = X
    y_all[i] = y


In [8]:
#random forest regressor model
results = {}
#for training cluade told me a good way to train the model is to leave one subject out and train and the rest and then test on the subject you left out (LOSO cross-validation)
for test_subject in range(1,9):
  x_train = np.vstack([X_all[i] for i in range(1,9) if i != test_subject])
  y_train = np.concatenate([y_all[i] for i in range(1,9) if i != test_subject])
  x_train, y_train = shuffle(x_train, y_train, random_state=50)
  
  model = RandomForestClassifier(n_estimators=100, random_state=50, n_jobs=-1)
  model.fit(x_train, y_train)
  y_pred = model.predict(X_all[test_subject])
  acc = accuracy_score(y_all[test_subject], y_pred)
  
  results[test_subject] = {"accuracy": acc, "report": classification_report(y_all[test_subject], y_pred)}
  
  print(f"Subject {test_subject} (test) — Accuracy: {acc:.4f}")
  
  
print(f"\nMean LOSO Accuracy: {np.mean([r['accuracy'] for r in results.values()]):.4f}")


Subject 1 (test) — Accuracy: 0.8006
Subject 2 (test) — Accuracy: 0.9494
Subject 3 (test) — Accuracy: 0.9556
Subject 4 (test) — Accuracy: 0.9540
Subject 5 (test) — Accuracy: 0.9529
Subject 6 (test) — Accuracy: 0.9292
Subject 7 (test) — Accuracy: 0.9656
Subject 8 (test) — Accuracy: 0.9666

Mean LOSO Accuracy: 0.9342
